In [8]:
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("ReadParquetAndStreamToKafka") \
    .getOrCreate()

# Read Parquet file into a DataFrame
df = spark.read.parquet("data/user_scores.parquet")

# Show the DataFrame schema and content
df.printSchema()
df.show(truncate=False)


root
 |-- Game Name: string (nullable = true)
 |-- Score: double (nullable = true)
 |-- Plays: string (nullable = true)
 |-- Genres: string (nullable = true)

+---------------------------------------+-----------------+-----+------------------------------------------+
|Game Name                              |Score            |Plays|Genres                                    |
+---------------------------------------+-----------------+-----+------------------------------------------+
|Elden Ring                             |90.0             |50K  |Adventure, RPG                            |
|Hades                                  |NULL             |4    |Shooter                                   |
|The Legend of Zelda: Breath of the Wild|88.00000000000001|66K  |Adventure, Puzzle, RPG                    |
|Undertale                              |86.0             |58K  |Adventure, Indie, RPG, Turn Based Strategy|
|Hollow Knight                          |88.00000000000001|49K  |Adventure, In

In [10]:
# Write data to Kafka topic
kafka_broker = "localhost:29092"
kafka_topic = "user_scores_topic"

df.withColumnRenamed("Game Name", "GameName") \
  .selectExpr("CAST(GameName AS STRING) AS key", "to_json(struct(*)) AS value") \
  .write \
  .format("kafka") \
  .option("kafka.bootstrap.servers", kafka_broker) \
  .option("topic", kafka_topic) \
  .save()


AnalysisException: Failed to find data source: kafka. Please deploy the application as per the deployment section of Structured Streaming + Kafka Integration Guide.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, from_json
from pyspark.sql.types import StructType, StringType, DoubleType

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("KafkaSparkStreaming") \
    .getOrCreate()

# Define schema for Kafka message
schema = StructType() \
    .add("Game Name", StringType()) \
    .add("Score", DoubleType()) \
    .add("Plays", StringType()) \
    .add("Genres", StringType())

# Read from Kafka
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:29092") \
    .option("subscribe", "user_scores_topic") \
    .load()

# Convert value from binary to string
kafka_stream = kafka_stream.selectExpr("CAST(value AS STRING)")

# Parse JSON data
parsed_stream = kafka_stream.select(from_json(col("value"), schema).alias("data")).select("data.*")

# Drop NA values
parsed_stream = parsed_stream.dropna()

# Round the scores
parsed_stream = parsed_stream.withColumn("Score", expr("ROUND(Score)").cast(IntegerType()))

# Function to process plays count
def process_plays(plays):
    if 'K' in plays:
        return int(float(plays.replace('K', '')) * 1000)
    return int(plays)

# Apply the function to Plays column
processed_stream = parsed_stream.withColumn("Plays", expr("process_plays(Plays)").cast(IntegerType()))

# Write processed data to Parquet file
query = processed_stream.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("checkpointLocation", "checkpoints/user_scores") \
    .start("data/cleaned_user_scores.parquet")

# Await termination
query.awaitTermination()
